# ============================================================
# Notebook 07 :: Gold Feature Engineering
# India Flood Risk Platform
# ============================================================

In [1]:
# ============================================================
# Libraries
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

SILVER_DIR = DATA_DIR / "silver"

print("Project Root :", PROJECT_ROOT)
print("Silver Folder :", SILVER_DIR)

Project Root : d:\Work\Projects\India_Flood_Risk_Platform
Silver Folder : d:\Work\Projects\India_Flood_Risk_Platform\data\silver


# Gold Feature Engineering

## Objective

The Gold Layer transforms standardized rainfall observations and historical flood event information into an analytical dataset suitable for flood risk modelling.

The primary objectives are:

- Engineer hazard-related rainfall features.
- Incorporate temporal information.
- Integrate historical flood exposure.
- Generate a modelling-ready dataset for actuarial flood risk assessment.

Outputs from this notebook will serve as inputs for statistical modelling, Extreme Value Theory, return period estimation, and dashboard development.

In [5]:
# ============================================================
# Load Datasets
# ============================================================

rainfall = pd.read_parquet(
    DATA_DIR / "bronze" / "bronze_rainfall_master.parquet"
)

grid = pd.read_parquet(
    SILVER_DIR / "silver_grid_master.parquet"
)

events = pd.read_parquet(
    SILVER_DIR / "silver_emdat_events.parquet"
)

In [6]:
# ============================================================
# Validate Input Datasets
# ============================================================

print("=" * 60)
print("RAINFALL")
print("=" * 60)
print(rainfall.info())

print("\n")

print("=" * 60)
print("GRID")
print("=" * 60)
print(grid.info())

print("\n")

print("=" * 60)
print("EVENTS")
print("=" * 60)
print(events.info())

RAINFALL
<class 'pandas.DataFrame'>
RangeIndex: 88567160 entries, 0 to 88567159
Data columns (total 5 columns):
 #   Column       Dtype         
---  ------       -----         
 0   date         datetime64[ns]
 1   latitude     float64       
 2   longitude    float64       
 3   number       int64         
 4   rainfall_mm  float32       
dtypes: datetime64[ns](1), float32(1), float64(2), int64(1)
memory usage: 3.0 GB
None


GRID
<class 'pandas.DataFrame'>
RangeIndex: 15609 entries, 0 to 15608
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   grid_id    15609 non-null  int64  
 1   latitude   15609 non-null  float64
 2   longitude  15609 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 366.0 KB
None


EVENTS
<class 'pandas.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  

In [7]:
rainfall.head()

,date,latitude,longitude,number,rainfall_mm
0,2000-01-01,37.0,68.00,0,0.0
1,2000-01-01,37.0,68.25,0,0.0
2,2000-01-01,37.0,68.50,0,0.0
3,2000-01-01,37.0,68.75,0,0.0
4,2000-01-01,37.0,69.00,0,0.0


In [8]:
# ============================================================
# Attach Grid IDs to Rainfall Dataset
# ============================================================

gold = rainfall.merge(
    grid,
    on=["latitude", "longitude"],
    how="left"
)

print("=" * 60)
print("GRID ID MERGE")
print("=" * 60)

print("Rows :", len(gold))
print("Missing Grid IDs :", gold["grid_id"].isna().sum())

gold.head()

GRID ID MERGE
Rows : 88567160
Missing Grid IDs : 0


,date,latitude,longitude,number,rainfall_mm,grid_id
0,2000-01-01,37.0,68.00,0,0.0,15005
1,2000-01-01,37.0,68.25,0,0.0,15006
2,2000-01-01,37.0,68.50,0,0.0,15007
3,2000-01-01,37.0,68.75,0,0.0,15008
4,2000-01-01,37.0,69.00,0,0.0,15009


In [9]:
print("Unique Grid IDs :", gold["grid_id"].nunique())

Unique Grid IDs : 15609


In [10]:
# ============================================================
# Organize Gold Dataset
# ============================================================

gold = gold[
    [
        "date",
        "grid_id",
        "latitude",
        "longitude",
        "rainfall_mm"
    ]
]

gold = gold.sort_values(
    ["grid_id", "date"]
).reset_index(drop=True)

gold.head()

,date,grid_id,latitude,longitude,rainfall_mm
0,2000-01-01,1,6.0,68.0,0.416279
1,2000-01-02,1,6.0,68.0,0.000000
2,2000-01-03,1,6.0,68.0,0.343323
3,2000-01-04,1,6.0,68.0,0.214100
4,2000-01-05,1,6.0,68.0,0.065327


In [11]:
print(gold.shape)

print(gold.columns)

gold.head()

(88567160, 5)
Index(['date', 'grid_id', 'latitude', 'longitude', 'rainfall_mm'], dtype='str')


,date,grid_id,latitude,longitude,rainfall_mm
0,2000-01-01,1,6.0,68.0,0.416279
1,2000-01-02,1,6.0,68.0,0.000000
2,2000-01-03,1,6.0,68.0,0.343323
3,2000-01-04,1,6.0,68.0,0.214100
4,2000-01-05,1,6.0,68.0,0.065327


In [12]:
# ============================================================
# Feature Engineering
# 3-Day Cumulative Rainfall
# ============================================================

gold["rainfall_3d"] = (

    gold
    .groupby("grid_id")["rainfall_mm"]
    .transform(
        lambda x: x.rolling(
            window=3,
            min_periods=1
        ).sum()
    )

)

gold[
    [
        "grid_id",
        "date",
        "rainfall_mm",
        "rainfall_3d"
    ]
].head(15)

,grid_id,date,rainfall_mm,rainfall_3d
0,1,2000-01-01,0.416279,0.416279
1,1,2000-01-02,0.000000,0.416279
2,1,2000-01-03,0.343323,0.759602
3,1,2000-01-04,0.214100,0.557423
4,1,2000-01-05,0.065327,0.622749
...,...,...,...,...
10,1,2000-01-11,22.819996,48.218727
11,1,2000-01-12,18.953323,58.767319
12,1,2000-01-13,2.272844,44.046164
13,1,2000-01-14,9.148598,30.374765


In [13]:
gold[
    gold["grid_id"] == gold["grid_id"].iloc[0]
][
    [
        "date",
        "rainfall_mm",
        "rainfall_3d"
    ]
].head(10)

,date,rainfall_mm,rainfall_3d
0,2000-01-01,0.416279,0.416279
1,2000-01-02,0.000000,0.416279
2,2000-01-03,0.343323,0.759602
3,2000-01-04,0.214100,0.557423
4,2000-01-05,0.065327,0.622749
5,2000-01-06,0.322342,0.601768
6,2000-01-07,1.462936,1.850605
7,2000-01-08,8.253574,10.038853
8,2000-01-09,8.404732,18.121243
9,2000-01-10,16.993999,33.652306


In [14]:
# ============================================================
# 7-Day Cumulative Rainfall
# ============================================================

gold["rainfall_7d"] = (

    gold
    .groupby("grid_id")["rainfall_mm"]
    .transform(
        lambda x: x.rolling(
            window=7,
            min_periods=1
        ).sum()
    )

)

In [15]:
# ============================================================
# 30-Day Cumulative Rainfall
# ============================================================

gold["rainfall_30d"] = (

    gold
    .groupby("grid_id")["rainfall_mm"]
    .transform(
        lambda x: x.rolling(
            window=30,
            min_periods=1
        ).sum()
    )

)

In [16]:
gold.loc[
    gold["grid_id"] == 1,
    [
        "date",
        "rainfall_mm",
        "rainfall_3d",
        "rainfall_7d",
        "rainfall_30d"
    ]
].head(15)

,date,rainfall_mm,rainfall_3d,rainfall_7d,rainfall_30d
0,2000-01-01,0.416279,0.416279,0.416279,0.416279
1,2000-01-02,0.000000,0.416279,0.416279,0.416279
2,2000-01-03,0.343323,0.759602,0.759602,0.759602
3,2000-01-04,0.214100,0.557423,0.973701,0.973701
4,2000-01-05,0.065327,0.622749,1.039028,1.039028
...,...,...,...,...,...
10,2000-01-11,22.819996,48.218727,58.322906,59.296608
11,2000-01-12,18.953323,58.767319,77.210903,78.249931
12,2000-01-13,2.272844,44.046164,79.161406,80.522776
13,2000-01-14,9.148598,30.374765,86.847067,89.671373


In [17]:
# ============================================================
# Maximum Rainfall (Previous 7 Days)
# ============================================================

gold["max_rainfall_7d"] = (

    gold
    .groupby("grid_id")["rainfall_mm"]
    .transform(
        lambda x: x.rolling(
            window=7,
            min_periods=1
        ).max()
    )

)

In [18]:
# ============================================================
# Average Rainfall (Previous 30 Days)
# ============================================================

gold["mean_rainfall_30d"] = (

    gold
    .groupby("grid_id")["rainfall_mm"]
    .transform(
        lambda x: x.rolling(
            window=30,
            min_periods=1
        ).mean()
    )

)

In [19]:
gold.loc[
    gold["grid_id"] == 1,
    [
        "date",
        "rainfall_mm",
        "rainfall_3d",
        "rainfall_7d",
        "rainfall_30d",
        "max_rainfall_7d",
        "mean_rainfall_30d"
    ]
].head(15)

,date,rainfall_mm,rainfall_3d,rainfall_7d,rainfall_30d,max_rainfall_7d,mean_rainfall_30d
0,2000-01-01,0.416279,0.416279,0.416279,0.416279,0.416279,0.416279
1,2000-01-02,0.000000,0.416279,0.416279,0.416279,0.416279,0.208139
2,2000-01-03,0.343323,0.759602,0.759602,0.759602,0.416279,0.253201
3,2000-01-04,0.214100,0.557423,0.973701,0.973701,0.416279,0.243425
4,2000-01-05,0.065327,0.622749,1.039028,1.039028,0.416279,0.207806
...,...,...,...,...,...,...,...
10,2000-01-11,22.819996,48.218727,58.322906,59.296608,22.819996,5.390601
11,2000-01-12,18.953323,58.767319,77.210903,78.249931,22.819996,6.520828
12,2000-01-13,2.272844,44.046164,79.161406,80.522776,22.819996,6.194060
13,2000-01-14,9.148598,30.374765,86.847067,89.671373,22.819996,6.405098


In [20]:
# ============================================================
# Calendar Features
# ============================================================

gold["year"] = gold["date"].dt.year

In [21]:
gold["month"] = gold["date"].dt.month

In [22]:
gold["day_of_year"] = gold["date"].dt.dayofyear

In [23]:
gold["monsoon"] = (
    gold["month"]
    .isin([6, 7, 8, 9])
    .astype(int)
)

In [24]:
gold[
    [
        "date",
        "year",
        "month",
        "day_of_year",
        "monsoon"
    ]
].sample(10).sort_values("date")

,date,year,month,day_of_year,monsoon
64389638,2001-03-22,2001,3,81,0
42702676,2001-07-22,2001,7,203,1
36864618,2001-10-28,2001,10,301,0
5266654,2003-04-25,2003,4,115,0
27205790,2005-05-24,2005,5,144,0
70324420,2009-10-08,2009,10,281,0
69033048,2010-03-09,2010,3,68,0
25227241,2012-06-03,2012,6,155,1
5749551,2013-05-31,2013,5,151,0
22568754,2013-11-18,2013,11,322,0


In [25]:
event_mapping = pd.read_parquet(
    SILVER_DIR / "silver_event_grid_mapping.parquet"
)

event_mapping.columns

Index(['event_id', 'grid_id', 'latitude', 'longitude', 'start_date',
       'end_date'],
      dtype='str')

# Gold Layer Progress Report

**Project:** India Flood Risk Platform
**Notebook:** 07_Gold_Feature_Engineering.ipynb
**Date:** 16 July 2026

---

# Objective

Continue the Gold Layer by engineering modelling-ready features from the rainfall dataset and finalize the modelling strategy for the Flood Risk Prediction pipeline.

---

# Work Completed

## 1. Rainfall Feature Engineering

Created temporal rainfall accumulation features to capture antecedent rainfall conditions responsible for flood generation.

### Features Created

| Feature             | Description                                                |
| ------------------- | ---------------------------------------------------------- |
| `rainfall_3d`       | Cumulative rainfall over the previous 3 days               |
| `rainfall_7d`       | Cumulative rainfall over the previous 7 days               |
| `rainfall_30d`      | Cumulative rainfall over the previous 30 days              |
| `max_rainfall_7d`   | Maximum daily rainfall observed within the previous 7 days |
| `mean_rainfall_30d` | Average daily rainfall during the previous 30 days         |

---

## 2. Feature Validation

Each rolling feature was manually verified using sample observations.

Validation confirmed:

* Correct rolling window implementation.
* Rolling calculations performed independently within each `grid_id`.
* No data leakage across spatial grids.
* Rolling windows behaved as expected for increasing temporal horizons.

---

## 3. Interpretation of Engineered Features

Rather than treating engineered variables as mathematical transformations, each feature was linked to its hydrological significance.

Established interpretations include:

* **3-Day Rainfall:** Short-term rainfall accumulation representing immediate runoff potential.
* **7-Day Rainfall:** Medium-term catchment saturation.
* **30-Day Rainfall:** Antecedent wetness and long-term moisture conditions.
* **Maximum 7-Day Rainfall:** Indicator of extreme rainfall intensity.
* **Mean 30-Day Rainfall:** Background climatic rainfall conditions.

These engineered variables provide meaningful physical explanations of flood generation processes.

---

## 4. Calendar Feature Engineering

Seasonality variables were extracted directly from the observation date.

### Features Created

| Feature       | Description                                        |
| ------------- | -------------------------------------------------- |
| `year`        | Observation year                                   |
| `month`       | Observation month                                  |
| `day_of_year` | Julian day of the year                             |
| `monsoon`     | Binary indicator (June–September = 1, otherwise 0) |

---

## 5. Validation of Calendar Features

Random observations were sampled to verify:

* Correct year extraction.
* Correct month assignment.
* Accurate Julian day computation.
* Correct identification of monsoon months.

All validations passed successfully.

---

# Technical Understanding Developed

Today's work focused heavily on understanding the **purpose** behind each engineered feature rather than simply implementing code.

Key concepts discussed included:

* Difference between rainfall accumulation and rainfall intensity.
* Importance of antecedent rainfall for flood generation.
* Why rolling windows provide historical context to each observation.
* Physical interpretation of rainfall features in relation to soil saturation, runoff, and riverine flooding.
* Distinction between accumulated rainfall and background climatic conditions.

---

# Modelling Strategy Finalized

A major project decision was finalized.

Instead of directly implementing an advanced machine learning algorithm, the project will follow a structured modelling hierarchy.

## Final Modelling Pipeline

1. Logistic Regression (Statistical Baseline)
2. Random Forest (Machine Learning Benchmark)
3. XGBoost (Advanced Ensemble Model)

Each model will be evaluated using identical datasets and performance metrics to justify the final model selection.

---

# Long-Term Project Architecture

The overall project workflow has now been finalized.

```text
ERA5 + EM-DAT
        │
        ▼
Bronze Layer
        ▼
Silver Layer
        ▼
Gold Feature Engineering
        ▼
Target Variable Construction
        ▼
Logistic Regression
        ▼
Random Forest
        ▼
XGBoost
        ▼
Model Comparison
        ▼
Flood Risk Dashboard
        ▼
Future Extension:
Actuarial Pricing using GLMs
```

---

# Target Variable Design

The methodology for constructing the supervised learning target was finalized.

### Definition

A rainfall observation will receive:

* **`flood_event = 1`** if its `grid_id` and `date` fall within the official EM-DAT event duration (`start_date` to `end_date`) after spatial matching.

* **`flood_event = 0`** otherwise.

This approach preserves the complete duration of documented flood events without introducing artificial lead or lag periods.

---

# Event Mapping Validation

Verified that the spatial event mapping dataset contains all required variables:

* `event_id`
* `grid_id`
* `latitude`
* `longitude`
* `start_date`
* `end_date`

This confirms that the Gold Layer can be directly merged with the event mapping table using:

* `grid_id`
* `date`

---

# Next Session Objectives

1. Expand EM-DAT events into daily observations.
2. Generate the binary target variable (`flood_event`).
3. Merge event labels with the Gold Layer.
4. Validate positive and negative class labels.
5. Produce the final modelling dataset.
6. Begin exploratory analysis for Logistic Regression.

---

# Outcome

Today's session marked the transition from **feature engineering** to **predictive modelling design**.

The Gold Layer now contains hydrologically meaningful rainfall and seasonal features, and the complete modelling roadmap—from Logistic Regression through XGBoost and finally to an actuarial pricing extension—has been formally established. This provides a clear, defensible framework for the remainder of the India Flood Risk Platform project.
